# 第13章（三）：浮点数表示与运算 Floating Point Representation and Arithmetic## Cambridge A-Level Computer Science (9618)---### 本节学习目标 Learning Objectives1. 理解**浮点数 (floating point)** 的表示方法：**尾数 (mantissa)** + **指数 (exponent)**2. 掌握浮点数的**规格化 (normalization)**——正数和负数的规格化条件3. 能够进行浮点数的**加法、减法、乘法**运算4. 理解**精度 (precision)**、**溢出 (overflow)**、**下溢 (underflow)** 和**舍入误差 (rounding errors)**5. 通过交互式演示深入理解浮点数的内部表示---### 为什么需要浮点数？Why Do We Need Floating Point?在计算机中，整数 (integer) 只能表示没有小数点的数。但现实世界充满了小数：- 圆周率 pi = 3.14159...- 地球到太阳的距离 = 149,600,000 km- 电子质量 = 0.000000000000000000000000000000911 kg**定点数 (fixed-point)** 的问题是：小数点位置固定，无法同时表示非常大和非常小的数。**浮点数的核心思想**：让小数点"浮动"——就像科学计数法！```科学计数法:  3.14 x 10^0  =  3.14             1.496 x 10^8  =  149,600,000             9.11 x 10^-31 =  0.000...000911```在二进制中：**数值 = 尾数 (mantissa) x 2^指数 (exponent)**> **Cambridge 考试重点**：你需要能够在二进制浮点数和十进制之间相互转换，并进行规格化。

In [ ]:
# 环境配置 Environment Setupimport matplotlib.pyplot as pltimport matplotlibimport matplotlib.patches as mpatchesimport numpy as npfrom IPython.display import display, HTML, Markdownmatplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']matplotlib.rcParams['axes.unicode_minus'] = Falseplt.rcParams['figure.figsize'] = (12, 6)print('环境配置完成！Ready to explore floating point numbers!')

---## 一、二进制小数回顾 Binary Fractions Review在学习浮点数之前，我们需要先理解**二进制小数**是怎么工作的。### 1.1 二进制小数的权值就像十进制中小数点右边每一位的权值是 10^-1, 10^-2, 10^-3...，二进制中是：| 位置 | ... | 2^2 | 2^1 | 2^0 | . | 2^-1 | 2^-2 | 2^-3 | 2^-4 | ... ||------|-----|-----|-----|-----|---|------|------|------|------|-----|| 权值 | ... | 4 | 2 | 1 | . | 0.5 | 0.25 | 0.125 | 0.0625 | ... |**例子**：二进制 `101.101` = ?```1 x 2^2  = 40 x 2^1  = 01 x 2^0  = 1.1 x 2^-1 = 0.50 x 2^-2 = 01 x 2^-3 = 0.125-----------------合计 = 5.625```### 1.2 十进制小数转二进制方法：**整数部分除2取余，小数部分乘2取整**例如 6.75 转二进制：- 整数部分 6: 6/2=3...0, 3/2=1...1, 1/2=0...1 -> 110- 小数部分 0.75: 0.75x2=1.5(取1), 0.5x2=1.0(取1) -> .11- 结果: **110.11**

In [ ]:
# 演示：二进制小数的转换# Demo: Binary Fraction Conversiondef decimal_to_binary_fraction(num, int_bits=8, frac_bits=8):    # Decimal to binary (with fractional part)    is_negative = num < 0    num = abs(num)    int_part = int(num)    frac_part = num - int_part    # 整数部分    int_binary = bin(int_part)[2:] if int_part > 0 else '0'    # 小数部分    frac_binary = ''    seen = set()    for _ in range(frac_bits):        frac_part *= 2        if frac_part >= 1:            frac_binary += '1'            frac_part -= 1        else:            frac_binary += '0'        if frac_part == 0:            break    result = f"{int_binary}.{frac_binary}"    return ('-' if is_negative else '') + resultdef binary_fraction_to_decimal(binary_str):    # Binary fraction to decimal    if '.' in binary_str:        int_part, frac_part = binary_str.split('.')    else:        int_part, frac_part = binary_str, ''    # 整数部分    value = int(int_part, 2) if int_part else 0    # 小数部分    for i, bit in enumerate(frac_part):        value += int(bit) * (2 ** -(i + 1))    return value# 演示print("=" * 60)print("二进制小数转换演示")print("=" * 60)test_values = [5.625, 6.75, 0.1, 3.14159, 10.3]for val in test_values:    binary = decimal_to_binary_fraction(val)    back = binary_fraction_to_decimal(binary.lstrip('-'))    print(f"  {val:>10} -> {binary:<20} -> {back}")print(f"\n注意: 0.1 和 3.14159 在二进制中是无限循环小数！")print(f"   这就是浮点数精度误差的根本原因。")

---## 二、浮点数的结构 Floating Point Structure### 2.1 基本结构一个浮点数由两部分组成：```浮点数 = 尾数 (Mantissa) x 2^指数 (Exponent)|<-- 尾数 (m位) -->|<-- 指数 (e位) -->|| s x x x x x x x | s y y y y y y y |```- **尾数 (Mantissa)**：也叫有效数字 (significand)，决定数的**精度**  - 第一位是**符号位**：0 = 正数，1 = 负数  - 尾数用**二进制补码 (two's complement)** 表示  - 尾数存储的是一个 **-1 到 +1 之间的小数**（小数点在符号位之后）- **指数 (Exponent)**：决定小数点的位置，即数的**大小范围**  - 也用**二进制补码**表示  - 指数越大，数越大；指数越小（负数），数越接近零### 2.2 Cambridge A-Level 中的约定在 Cambridge 9618 考试中，通常：- 尾数和指数都用**二进制补码 (two's complement)** 表示- 尾数的小数点在**符号位之后**（即 `S.XXXXXXX` 的形式）- 例如尾数 `0.1101000`，表示 +0.1101 (二进制) = 0.8125 (十进制)### 2.3 举例假设 8 位尾数 + 4 位指数的浮点数：```尾数: 0.1101000   指数: 0100     = +0.8125          = +4数值 = 0.8125 x 2^4 = 0.8125 x 16 = 13.0```

In [ ]:
# 可视化：浮点数的结构# Visualization: Floating Point Number Structurefig, ax = plt.subplots(figsize=(14, 6))# 画主结构# 尾数部分mantissa_bits = ['0', '.', '1', '1', '0', '1', '0', '0', '0']colors_m = ['#FFD700', '#FFFFFF', '#90EE90', '#90EE90', '#90EE90', '#90EE90', '#90EE90', '#90EE90', '#90EE90']labels_m = ['S', '.', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7']for i, (bit, color, label) in enumerate(zip(mantissa_bits, colors_m, labels_m)):    if bit == '.':        ax.text(1.5 + i * 0.9, 4.5, '.', ha='center', va='center', fontsize=20, fontweight='bold', color='red')        continue    rect = mpatches.FancyBboxPatch((1 + i * 0.9, 4.1), 0.8, 0.8,        boxstyle="round,pad=0.05", facecolor=color, edgecolor='black', linewidth=2)    ax.add_patch(rect)    ax.text(1.4 + i * 0.9, 4.5, bit, ha='center', va='center', fontsize=16, fontweight='bold')    ax.text(1.4 + i * 0.9, 3.8, label, ha='center', va='center', fontsize=9, color='gray')ax.text(4.5, 5.2, '尾数 Mantissa (8 bits)', ha='center', fontsize=13, fontweight='bold', color='#006600')# 指数部分exponent_bits = ['0', '1', '0', '0']for i, bit in enumerate(exponent_bits):    x = 10 + i * 0.9    color = '#FFD700' if i == 0 else '#ADD8E6'    rect = mpatches.FancyBboxPatch((x, 4.1), 0.8, 0.8,        boxstyle="round,pad=0.05", facecolor=color, edgecolor='black', linewidth=2)    ax.add_patch(rect)    ax.text(x + 0.4, 4.5, bit, ha='center', va='center', fontsize=16, fontweight='bold')ax.text(11.5, 5.2, '指数 Exponent (4 bits)', ha='center', fontsize=13, fontweight='bold', color='#000099')# 计算过程ax.text(1, 2.8, '计算过程:', fontsize=13, fontweight='bold')ax.text(1, 2.3, '尾数 = 0.1101000 (二进制) = 1/2 + 1/4 + 1/16 = 0.8125', fontsize=11)ax.text(1, 1.8, '指数 = 0100 (二进制) = +4', fontsize=11)ax.text(1, 1.3, '数值 = 0.8125 x 2^4 = 0.8125 x 16 = 13.0', fontsize=12, fontweight='bold', color='#CC0000')# 符号位说明ax.annotate('符号位\n(0=正)', xy=(1.4, 4.1), xytext=(0.5, 3.3),            fontsize=10, ha='center', color='#B8860B',            arrowprops=dict(arrowstyle='->', color='#B8860B', lw=1.5))ax.set_xlim(0, 15); ax.set_ylim(0.8, 5.8)ax.set_title('浮点数结构: 尾数 + 指数', fontsize=15, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()

---## 三、规格化 Normalization### 3.1 为什么需要规格化？同一个数可以有多种浮点表示：```13.0 = 0.8125  x 2^4    尾数: 0.1101000, 指数: 010013.0 = 0.40625 x 2^5    尾数: 0.0110100, 指数: 010113.0 = 1.625   x 2^3    尾数: 0.0011010, 指数: 0011  (不对，超出范围了)```如果不规定统一的格式，同一个数就会有多种表示方法，这会导致：1. **浪费精度**：尾数前面的0不携带任何信息2. **比较困难**：无法直接比较两个浮点数### 3.2 规格化的规则**规格化 (Normalization)** 要求尾数的**最高有效位**（符号位之后的第一位）必须与符号位**不同**。**正数规格化**：尾数形如 `0.1xxxxxx`- 符号位是 0，下一位必须是 1- 这保证尾数 >= 0.5 且 < 1.0**负数规格化**：尾数形如 `1.0xxxxxx`- 符号位是 1（负数），下一位必须是 0- 在二进制补码中，这保证尾数 >= -1.0 且 < -0.5### 3.3 规格化的过程**如果尾数太小**（前面有多余的0或1）：- 正数：尾数**左移**，指数**减小**- 负数：尾数**左移**，指数**减小**例如：```未规格化: 尾数 = 0.0110100, 指数 = 0101 (5)左移一位: 尾数 = 0.1101000, 指数 = 0100 (4)  <-- 规格化完成！```> **Cambridge 考试陷阱**：很多学生会忘记负数的规格化条件！记住：负数是 `1.0xxxxxx`，不是 `1.1xxxxxx`。

In [ ]:
# 演示：规格化过程# Demo: Normalization Processdef normalize_float(mantissa_bits, exponent_val, m_size=8):    """规格化浮点数    mantissa_bits: 尾数的位列表，如 [0,0,1,1,0,1,0,0]    exponent_val: 指数的十进制值    m_size: 尾数位数    """    m = mantissa_bits.copy()    exp = exponent_val    steps = []    sign_bit = m[0]    # 检查是否已经规格化    if len(m) > 1:        if sign_bit == 0 and m[1] == 1:            steps.append("已经是规格化的正数 (0.1...)")            return m, exp, steps        elif sign_bit == 1 and m[1] == 0:            steps.append("已经是规格化的负数 (1.0...)")            return m, exp, steps    # 需要规格化：左移    shift_count = 0    while len(m) > 1 and m[0] == m[1]:        # 左移：去掉第二位，末尾补符号位的值        removed = m.pop(1)        if sign_bit == 0:            m.append(0)        else:            m.append(1)        exp -= 1        shift_count += 1        steps.append(f"左移第{shift_count}次: 尾数={''.join(map(str,m))}, 指数={exp}")        # 检查是否规格化        if m[0] != m[1]:            steps.append(f"规格化完成！符号位={m[0]}, 下一位={m[1]} (不同)")            break    return m, exp, steps# 示例1：正数规格化print("=" * 60)print("示例1: 正数规格化")print("=" * 60)m1 = [0, 0, 1, 1, 0, 1, 0, 0]e1 = 5print(f"原始: 尾数 = {'.'.join([''.join(map(str,m1[:1])), ''.join(map(str,m1[1:]))])}, 指数 = {e1}")m1_norm, e1_norm, steps1 = normalize_float(m1, e1)for s in steps1:    print(f"  {s}")print(f"结果: 尾数 = {m1_norm[0]}.{''.join(map(str,m1_norm[1:]))}, 指数 = {e1_norm}")# 示例2：负数规格化print(f"\n{'=' * 60}")print("示例2: 负数规格化")print("=" * 60)m2 = [1, 1, 0, 1, 0, 1, 0, 0]e2 = 3print(f"原始: 尾数 = {m2[0]}.{''.join(map(str,m2[1:]))}, 指数 = {e2}")m2_norm, e2_norm, steps2 = normalize_float(m2, e2)for s in steps2:    print(f"  {s}")print(f"结果: 尾数 = {m2_norm[0]}.{''.join(map(str,m2_norm[1:]))}, 指数 = {e2_norm}")# 示例3：已经规格化print(f"\n{'=' * 60}")print("示例3: 已经规格化的数")print("=" * 60)m3 = [0, 1, 1, 0, 1, 0, 0, 0]e3 = 4print(f"原始: 尾数 = {m3[0]}.{''.join(map(str,m3[1:]))}, 指数 = {e3}")m3_norm, e3_norm, steps3 = normalize_float(m3, e3)for s in steps3:    print(f"  {s}")

In [ ]:
# 可视化：规格化过程动画示意# Visualization: Normalization Processfig, axes = plt.subplots(3, 1, figsize=(14, 8))def draw_bits(ax, bits, exp, title, highlight_pos=None, is_normalized=False):    ax.set_xlim(-0.5, 14)    ax.set_ylim(-0.3, 1.5)    ax.axis('off')    # 标题    status = " [已规格化]" if is_normalized else " [未规格化]"    color = '#006600' if is_normalized else '#CC0000'    ax.set_title(title + status, fontsize=12, fontweight='bold', color=color, loc='left')    # 尾数位    for i, bit in enumerate(bits):        x = i * 0.9        if i == 0:            fc = '#FFD700'  # 符号位        elif highlight_pos and i in highlight_pos:            fc = '#FF6B6B'  # 高亮        elif i == 1:            fc = '#90EE90' if (bits[0] != bit) else '#FFCCCC'        else:            fc = '#E0E0E0'        rect = mpatches.FancyBboxPatch((x, 0.3), 0.75, 0.7,            boxstyle="round,pad=0.05", facecolor=fc, edgecolor='black', linewidth=1.5)        ax.add_patch(rect)        ax.text(x + 0.375, 0.65, str(bit), ha='center', va='center', fontsize=14, fontweight='bold')    # 小数点    ax.text(0.85, 0.5, '.', fontsize=20, fontweight='bold', color='red')    # 指数    ax.text(9, 0.65, f'Exponent = {exp}', fontsize=13, fontweight='bold', color='#000099',            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E0E0FF', edgecolor='#000099'))# 未规格化draw_bits(axes[0], [0,0,1,1,0,1,0,0], 5, "Step 0: 原始")# 左移一次draw_bits(axes[1], [0,1,1,0,1,0,0,0], 4, "Step 1: 左移1位", highlight_pos={1})# 规格化完成draw_bits(axes[2], [0,1,1,0,1,0,0,0], 4, "Step 2: 完成", is_normalized=True)# 箭头for i in range(2):    fig.patches.append(mpatches.FancyArrowPatch(        (0.5, 0.63 - i * 0.33), (0.5, 0.37 - i * 0.33),        transform=fig.transFigure, arrowstyle='->', mutation_scale=20,        color='#333', linewidth=2))plt.suptitle('正数规格化过程: 0.0110100 x 2^5 -> 0.1101000 x 2^4', fontsize=14, fontweight='bold')plt.tight_layout(rect=[0, 0, 1, 0.93])plt.show()

---## 四、浮点数转换 Floating Point Conversion### 4.1 二进制浮点数 -> 十进制**步骤**：1. 读取尾数（二进制补码），转换为十进制小数2. 读取指数（二进制补码），转换为十进制整数3. 计算：数值 = 尾数 x 2^指数### 4.2 十进制 -> 二进制浮点数**步骤**：1. 将十进制数转换为二进制小数2. 将二进制小数写成 `0.1xxxx x 2^n` 的规格化形式3. 分别存储尾数和指数（二进制补码）### 4.3 负数的尾数（二进制补码）对于负数，尾数用二进制补码表示。回顾补码规则：- **正数**的补码 = 原码本身- **负数**的补码 = 取反 + 1例如 -0.75：```+0.75 = 0.1100000取反  = 1.0011111+1    = 1.0100000   <-- 这就是 -0.75 的补码表示```验证：`1.0100000` 是规格化的吗？符号位=1，下一位=0 -> 是！

In [ ]:
# 交互式浮点数转换器# Interactive Floating Point Converterclass FloatingPointConverter:    """浮点数转换器 - 模拟 Cambridge A-Level 的浮点数表示"""    def __init__(self, mantissa_bits=8, exponent_bits=4):        self.m_bits = mantissa_bits        self.e_bits = exponent_bits    def twos_complement_to_decimal(self, bits):        """二进制补码转十进制整数"""        if bits[0] == 0:            return int(''.join(map(str, bits)), 2)        else:            # 负数：取反+1            inverted = [1 - b for b in bits]            val = int(''.join(map(str, inverted)), 2) + 1            return -val    def mantissa_to_decimal(self, m_bits):        """Mantissa (two's complement fraction) to decimal"""        if m_bits[0] == 0:            # 正数            value = 0.0            for i in range(1, len(m_bits)):                value += m_bits[i] * (2 ** -i)            return value        else:            # 负数：用补码方法            # 先将整个尾数看作整数形式的补码            int_val = self.twos_complement_to_decimal(m_bits)            # 除以 2^(位数-1) 得到小数值            return int_val / (2 ** (len(m_bits) - 1))    def binary_float_to_decimal(self, mantissa, exponent):        """完整的二进制浮点数转十进制"""        m_val = self.mantissa_to_decimal(mantissa)        e_val = self.twos_complement_to_decimal(exponent)        result = m_val * (2 ** e_val)        return m_val, e_val, result    def is_normalized(self, mantissa):        """检查是否规格化"""        if len(mantissa) < 2:            return False        return mantissa[0] != mantissa[1]# 创建转换器fpc = FloatingPointConverter(mantissa_bits=8, exponent_bits=4)# 测试多个浮点数print("=" * 70)print("浮点数转换演示 (8位尾数 + 4位指数)")print("=" * 70)test_cases = [    ([0,1,1,0,1,0,0,0], [0,1,0,0], "正数示例 13.0"),    ([0,1,0,0,0,0,0,0], [0,0,0,1], "正数示例 1.0"),    ([0,1,1,0,0,0,0,0], [0,0,0,0], "正数示例 0.75"),    ([1,0,0,1,1,0,0,0], [0,1,0,0], "-13.0 (负数)"),    ([1,0,1,1,0,0,0,0], [0,0,1,0], "-5.0 (负数)"),    ([0,1,0,0,0,0,0,0], [1,1,0,1], "很小的正数"),]for mantissa, exponent, desc in test_cases:    m_val, e_val, result = fpc.binary_float_to_decimal(mantissa, exponent)    m_str = f"{mantissa[0]}.{''.join(map(str, mantissa[1:]))}"    e_str = ''.join(map(str, exponent))    normalized = "YES" if fpc.is_normalized(mantissa) else "NO"    print(f"\n  {desc}:")    print(f"    尾数 = {m_str} = {m_val}")    print(f"    指数 = {e_str} = {e_val}")    print(f"    数值 = {m_val} x 2^{e_val} = {result}")    print(f"    规格化? {normalized}")

---## 五、负数的浮点数表示 Negative Floating Point Numbers### 5.1 补码表示的尾数在 Cambridge A-Level 中，尾数用**二进制补码**表示。理解负数的尾数是一个常见难点。**关键要点**：| 尾数 | 十进制值 | 规格化？ | 解释 ||------|---------|---------|------|| `0.1000000` | +0.5 | YES | 正数，符号位后是1 || `0.1111111` | +0.9921875 | YES | 接近+1.0 || `1.0000000` | -1.0 | YES | 最小的负数尾数 || `1.0111111` | -0.5078125 | YES | 接近-0.5 || `1.1000000` | -0.5 | **NO!** | 符号位=1，下一位=1 -> 未规格化 || `0.0110000` | +0.375 | **NO!** | 符号位=0，下一位=0 -> 未规格化 |### 5.2 负数浮点数的例子**例：表示 -5.0**1. 先表示 +5.0 = 101.0 (二进制)2. 写成规格化形式: 0.101 x 2^33. 尾数 +0.101 的补码: `0.1010000`4. 取负: 取反+1 -> `1.0110000`5. 指数 = 3 = `0011`6. 验证: `1.0110000` 规格化？符号位=1, 下一位=0 -> YES!

In [ ]:
# 可视化：正数和负数的规格化范围# Visualization: Normalized Range for Positive and Negative Numbersfig, ax = plt.subplots(figsize=(14, 6))# 尾数的范围# 正数规格化: 0.1000000 到 0.1111111 -> [0.5, ~1.0)# 负数规格化: 1.0000000 到 1.0111111 -> [-1.0, ~-0.5)y = 0.5# 画数轴ax.axhline(y=y, color='black', linewidth=2, xmin=0.05, xmax=0.95)# 标记点marks = [    (-1.0, '1.0000000\n(-1.0)', 'red'),    (-0.75, '1.0100000\n(-0.75)', 'red'),    (-0.5, '1.1000000\n(-0.5)\nNOT normalized!', '#999'),    (-0.25, '', '#999'),    (0, '0', 'black'),    (0.25, '', '#999'),    (0.5, '0.1000000\n(+0.5)', 'green'),    (0.75, '0.1100000\n(+0.75)', 'green'),    (1.0, '', '#999'),]for val, label, color in marks:    x = 0.5 + val * 0.4  # scale to plot    ax.plot(x, y, 'o', color=color, markersize=10, zorder=5)    if label:        y_offset = 0.15 if val >= 0 else -0.25        ax.text(x, y + y_offset, label, ha='center', va='center' if val >= 0 else 'top',                fontsize=9, fontweight='bold', color=color)# 规格化范围# 正数范围ax.annotate('', xy=(0.7, y - 0.08), xytext=(0.9, y - 0.08),            arrowprops=dict(arrowstyle='<->', color='green', lw=2))ax.text(0.8, y - 0.15, '正数规格化范围\n[+0.5, +1.0)', ha='center', fontsize=10, color='green', fontweight='bold')# 负数范围ax.annotate('', xy=(0.1, y - 0.08), xytext=(0.3, y - 0.08),            arrowprops=dict(arrowstyle='<->', color='red', lw=2))ax.text(0.2, y - 0.15, '负数规格化范围\n[-1.0, -0.5)', ha='center', fontsize=10, color='red', fontweight='bold')# 未规格化区域ax.axvspan(0.3, 0.5, alpha=0.1, color='gray')ax.axvspan(0.5, 0.7, alpha=0.1, color='gray')ax.text(0.5, y + 0.35, '灰色区域: 未规格化', ha='center', fontsize=11, color='gray',        bbox=dict(boxstyle='round', facecolor='#F0F0F0', edgecolor='gray'))ax.set_xlim(0, 1)ax.set_ylim(-0.1, 1.0)ax.set_title('尾数 (Mantissa) 的规格化范围', fontsize=14, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()

---## 六、浮点数运算 Floating Point Arithmetic### 6.1 加法和减法浮点数加减法的核心步骤：1. **对齐指数**：将较小指数的数的尾数**右移**，使两个数的指数相同2. **尾数相加/相减**3. **规格化**结果4. 检查是否**溢出或下溢****例：计算 5.0 + 0.75**```5.0  = 0.1010000 x 2^3    (尾数=0.625, 指数=3)0.75 = 0.1100000 x 2^0    (尾数=0.75,  指数=0)Step 1: 对齐指数（将0.75的尾数右移3位）  0.75 -> 0.0001100 x 2^3Step 2: 尾数相加  0.1010000+ 0.0001100-----------  0.1011100Step 3: 结果 = 0.1011100 x 2^3  = 0.71875 x 8 = 5.75  (正确！)```### 6.2 乘法浮点数乘法相对简单：1. **尾数相乘**2. **指数相加**3. **规格化**结果```数值 = (m1 x 2^e1) x (m2 x 2^e2) = (m1 x m2) x 2^(e1 + e2)```

In [ ]:
# 演示：浮点数加法的详细过程# Demo: Detailed Floating Point Addition Processdef float_addition_demo(m1, e1, m2, e2, m_size=8):    """演示浮点数加法过程"""    print(f"  操作数1: 尾数={m1[0]}.{''.join(map(str,m1[1:]))}, 指数={e1}")    print(f"  操作数2: 尾数={m2[0]}.{''.join(map(str,m2[1:]))}, 指数={e2}")    # 计算十进制值    fpc = FloatingPointConverter()    val1_m = fpc.mantissa_to_decimal(m1)    val2_m = fpc.mantissa_to_decimal(m2)    val1 = val1_m * (2 ** e1)    val2 = val2_m * (2 ** e2)    print(f"  十进制: {val1} + {val2} = {val1 + val2}")    # Step 1: 对齐指数    print(f"\n  Step 1: 对齐指数")    diff = abs(e1 - e2)    if e1 > e2:        print(f"    操作数2的指数较小，尾数右移 {diff} 位")        shifted_m2 = m2.copy()        for _ in range(diff):            shifted_m2.pop()            shifted_m2.insert(0, m2[0])  # 算术右移：用符号位填充        e2 = e1        print(f"    对齐后: 尾数={shifted_m2[0]}.{''.join(map(str,shifted_m2[1:]))}, 指数={e2}")        m2_aligned = shifted_m2        m1_aligned = m1.copy()    elif e2 > e1:        print(f"    操作数1的指数较小，尾数右移 {diff} 位")        shifted_m1 = m1.copy()        for _ in range(diff):            shifted_m1.pop()            shifted_m1.insert(0, m1[0])        e1 = e2        print(f"    对齐后: 尾数={shifted_m1[0]}.{''.join(map(str,shifted_m1[1:]))}, 指数={e1}")        m1_aligned = shifted_m1        m2_aligned = m2.copy()    else:        print(f"    指数相同，无需对齐")        m1_aligned = m1.copy()        m2_aligned = m2.copy()    # Step 2: 尾数相加    print(f"\n  Step 2: 尾数相加")    print(f"    {m1_aligned[0]}.{''.join(map(str,m1_aligned[1:]))}")    print(f"  + {m2_aligned[0]}.{''.join(map(str,m2_aligned[1:]))}")    print(f"    {'─' * (m_size + 1)}")    # 实际加法（简化：转十进制加）    v1 = fpc.mantissa_to_decimal(m1_aligned)    v2 = fpc.mantissa_to_decimal(m2_aligned)    result_m_val = v1 + v2    result_exp = e1    # 将结果转回二进制尾数（近似）    result_bits = []    if result_m_val >= 0:        result_bits.append(0)        temp = result_m_val    else:        result_bits.append(1)        temp = result_m_val + 1.0  # 简化处理    for i in range(1, m_size):        threshold = 2 ** -i        if result_m_val >= 0:            if temp >= threshold:                result_bits.append(1)                temp -= threshold            else:                result_bits.append(0)        else:            # 负数简化            result_bits.append(0)    print(f"  = {result_bits[0]}.{''.join(map(str,result_bits[1:]))}")    # Step 3: 规格化    print(f"\n  Step 3: 规格化检查")    if len(result_bits) > 1 and result_bits[0] != result_bits[1]:        print(f"    已经规格化！")    else:        print(f"    需要规格化（左移尾数，减小指数）")    result = result_m_val * (2 ** result_exp)    print(f"\n  最终结果: {result_m_val} x 2^{result_exp} = {result}")# 示例：5.0 + 0.75print("=" * 60)print("浮点数加法: 5.0 + 0.75")print("=" * 60)float_addition_demo(    [0,1,0,1,0,0,0,0], 3,   # 5.0 = 0.625 x 2^3    [0,1,1,0,0,0,0,0], 0    # 0.75 = 0.75 x 2^0)

In [ ]:
# 更详细的二进制加法演示# More detailed binary addition demodef binary_add_step_by_step(a_bits, b_bits):    # Binary twos complement addition, bit by bit    n = len(a_bits)    result = [0] * n    carry = 0    print(f"    逐位相加 (从右到左):")    for i in range(n - 1, -1, -1):        total = a_bits[i] + b_bits[i] + carry        result[i] = total % 2        carry = total // 2        pos_label = "符号位" if i == 0 else f"位{n-1-i}"        print(f"      [{pos_label}] {a_bits[i]} + {b_bits[i]} + carry({carry if i < n-1 else 0}) = {result[i]} (carry={carry})")    if carry:        print(f"    溢出 carry={carry} (在补码中通常丢弃)")    return resultprint("=" * 60)print("二进制尾数加法详解: 5.0 + 0.75")print("=" * 60)# 5.0 = 0.1010000 x 2^3# 0.75 对齐后 = 0.0001100 x 2^3print("\n对齐后的尾数:")a = [0, 1, 0, 1, 0, 0, 0, 0]b = [0, 0, 0, 0, 1, 1, 0, 0]print(f"  A = {a[0]}.{''.join(map(str, a[1:]))}  (5.0 的尾数)")print(f"  B = {b[0]}.{''.join(map(str, b[1:]))}  (0.75 对齐后)")print()result = binary_add_step_by_step(a, b)print(f"\n结果 = {result[0]}.{''.join(map(str, result[1:]))}")print(f"指数 = 3")# 验证fpc = FloatingPointConverter()m_val = fpc.mantissa_to_decimal(result)print(f"\n验证: {m_val} x 2^3 = {m_val * 8}")print(f"规格化? {'YES' if result[0] != result[1] else 'NO'}")

In [ ]:
# 演示：浮点数乘法# Demo: Floating Point Multiplicationprint("=" * 60)print("浮点数乘法: 3.0 x 2.5")print("=" * 60)# 3.0 = 0.1100000 x 2^2# 2.5 = 0.1010000 x 2^2m1_val = 0.75   # 0.1100000e1_val = 2m2_val = 0.625  # 0.1010000e2_val = 2print(f"\n操作数1: 3.0 = {m1_val} x 2^{e1_val}")print(f"操作数2: 2.5 = {m2_val} x 2^{e2_val}")print(f"\nStep 1: 尾数相乘")m_result = m1_val * m2_valprint(f"  {m1_val} x {m2_val} = {m_result}")print(f"\nStep 2: 指数相加")e_result = e1_val + e2_valprint(f"  {e1_val} + {e2_val} = {e_result}")print(f"\nStep 3: 中间结果")print(f"  {m_result} x 2^{e_result} = {m_result * (2**e_result)}")print(f"\nStep 4: 规格化")if m_result >= 0.5:    print(f"  尾数 {m_result} >= 0.5, 已规格化")else:    shifts = 0    while m_result < 0.5:        m_result *= 2        e_result -= 1        shifts += 1    print(f"  左移 {shifts} 位: 尾数={m_result}, 指数={e_result}")print(f"\n最终结果: {m_result} x 2^{e_result} = {m_result * (2**e_result)}")print(f"验证: 3.0 x 2.5 = {3.0 * 2.5}")

---## 七、精度、溢出和误差 Precision, Overflow, and Errors### 7.1 精度 Precision浮点数的精度由**尾数的位数**决定：- 尾数越多位 -> 能表示的有效数字越多 -> 精度越高- 但存储空间也越大**精度和范围的权衡**：| 分配方式 | 尾数 | 指数 | 精度 | 范围 ||---------|------|------|------|------|| 方案A | 12位 | 4位 | 高 | 小 || 方案B | 8位 | 8位 | 低 | 大 |### 7.2 溢出 Overflow当运算结果**太大**，超出了浮点数能表示的最大值时，就发生了**溢出 (overflow)**。- **正溢出**：结果大于能表示的最大正数- **负溢出**：结果小于能表示的最小负数### 7.3 下溢 Underflow当运算结果**太接近零**，小于浮点数能表示的最小正数时，就发生了**下溢 (underflow)**。下溢的结果通常被近似为 0，这会导致精度损失。### 7.4 舍入误差 Rounding Errors由于尾数位数有限，很多数无法精确表示，必须舍入。经典例子：十进制的 0.1 在二进制中是无限循环小数！```0.1 (十进制) = 0.000110011001100110011... (二进制，无限循环)```这就是为什么在很多编程语言中 `0.1 + 0.2 != 0.3`！

In [ ]:
# 演示：精度和舍入误差# Demo: Precision and Rounding Errorsprint("=" * 60)print("浮点数精度演示")print("=" * 60)# Python中经典的浮点数精度问题print("\n【Python 中的浮点数精度问题】")print(f"  0.1 + 0.2 = {0.1 + 0.2}")print(f"  0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")print(f"  差值 = {0.1 + 0.2 - 0.3}")print(f"  0.1 的精确二进制: 0.0001100110011001100110011...(循环)")# 不同尾数位数的精度对比print("\n【不同尾数位数的精度对比】")print(f"{'尾数位数':<10}{'表示 0.1 的近似值':<25}{'误差'}")print("-" * 55)for bits in [4, 6, 8, 10, 12, 16, 24]:    # 模拟有限位数的0.1表示    approx = 0.0    frac = 0.1    for i in range(1, bits):        frac *= 2        if frac >= 1:            approx += 2 ** -i            frac -= 1    error = abs(0.1 - approx)    print(f"{bits:<10}{approx:<25.15f}{error:.15f}")# 溢出和下溢演示print("\n\n【溢出和下溢示例 (8位尾数 + 4位指数)】")print("指数范围: -8 到 +7 (4位补码)")print("最大正数: ~0.9961 x 2^7 = ~127.5")print("最小正数: ~0.5 x 2^-8 = ~0.00195")print()print("如果计算 100.0 x 2.0 = 200.0 -> 溢出！(超过127.5)")print("如果计算 0.003 / 2.0 = 0.0015 -> 下溢！(小于0.00195)")

In [ ]:
# 可视化：浮点数在数轴上的分布# Visualization: Floating Point Number Distribution on Number Linefig, axes = plt.subplots(2, 1, figsize=(14, 7))# 生成简化的浮点数（4位尾数 + 3位指数）def generate_float_numbers(m_bits=4, e_bits=3):    numbers = set()    for m_int in range(2**(m_bits)):        mantissa = []        for i in range(m_bits):            mantissa.append((m_int >> (m_bits - 1 - i)) & 1)        # 检查规格化        if len(mantissa) > 1 and mantissa[0] == mantissa[1]:            continue        # 计算尾数值        if mantissa[0] == 0:            m_val = sum(mantissa[i] * 2**(-i) for i in range(1, m_bits))        else:            # 负数补码            inv = [1 - b for b in mantissa]            pos_val = sum(inv[i] * 2**(-i) for i in range(1, m_bits)) + 2**(-(m_bits-1))            m_val = -pos_val        if m_val == 0:            continue        for e_int in range(2**e_bits):            exp_bits = []            for i in range(e_bits):                exp_bits.append((e_int >> (e_bits - 1 - i)) & 1)            if exp_bits[0] == 0:                e_val = e_int            else:                e_val = e_int - 2**e_bits            number = m_val * (2 ** e_val)            if abs(number) < 100:  # 限制范围                numbers.add(round(number, 10))    return sorted(numbers)# 正数部分pos_numbers = [n for n in generate_float_numbers() if n > 0]neg_numbers = [n for n in generate_float_numbers() if n < 0]# 画正数分布ax1 = axes[0]ax1.scatter(pos_numbers, [0]*len(pos_numbers), c='green', s=30, zorder=5, alpha=0.7)ax1.axhline(y=0, color='black', linewidth=1)ax1.set_title(f'正浮点数分布 (4位尾数+3位指数) - 共{len(pos_numbers)}个', fontsize=13, fontweight='bold')ax1.set_xlabel('数值')ax1.set_ylim(-0.5, 0.5)ax1.set_yticks([])# 标注密度变化ax1.annotate('靠近0的地方\n数字更密集', xy=(0.5, 0), xytext=(2, 0.3),            fontsize=10, arrowprops=dict(arrowstyle='->', color='red'),            color='red', fontweight='bold')ax1.annotate('远离0的地方\n数字更稀疏', xy=(30, 0), xytext=(35, 0.3),            fontsize=10, arrowprops=dict(arrowstyle='->', color='blue'),            color='blue', fontweight='bold')# 画完整数轴ax2 = axes[1]all_nums = generate_float_numbers()colors = ['red' if n < 0 else 'green' for n in all_nums]ax2.scatter(all_nums, [0]*len(all_nums), c=colors, s=20, zorder=5, alpha=0.7)ax2.axhline(y=0, color='black', linewidth=1)ax2.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')ax2.set_title(f'全部浮点数分布 - 共{len(all_nums)}个 (红=负, 绿=正)', fontsize=13, fontweight='bold')ax2.set_xlabel('数值')ax2.set_ylim(-0.5, 0.5)ax2.set_yticks([])# 标注ax2.text(0, 0.35, '注意: 0附近无法表示的\n"间隙"(下溢区域)', ha='center',         fontsize=10, color='purple', fontweight='bold',         bbox=dict(boxstyle='round', facecolor='#F0E0FF', edgecolor='purple'))plt.tight_layout()plt.show()print(f"总共可表示 {len(all_nums)} 个不同的浮点数")print(f"正数 {len(pos_numbers)} 个, 负数 {len(neg_numbers)} 个")print(f"最大正数: {max(pos_numbers)}")print(f"最小正数: {min(pos_numbers)}")print(f"最大负数: {max(neg_numbers)}")print(f"最小负数: {min(neg_numbers)}")

In [ ]:
# 可视化：精度 vs 范围的权衡# Visualization: Precision vs Range Tradeofffig, axes = plt.subplots(1, 2, figsize=(14, 5))# 方案A：多尾数位（高精度，小范围）nums_a = generate_float_numbers(m_bits=5, e_bits=2)pos_a = [n for n in nums_a if 0 < n < 10]axes[0].scatter(pos_a, [0]*len(pos_a), c='green', s=40, alpha=0.7)axes[0].axhline(y=0, color='black', linewidth=1)axes[0].set_title(f'方案A: 5位尾数+2位指数\n{len(pos_a)}个正数, 数字密集但范围小',                  fontsize=12, fontweight='bold')axes[0].set_xlabel('数值')axes[0].set_ylim(-0.5, 0.5); axes[0].set_yticks([])axes[0].set_xlim(-0.5, 10)# 方案B：多指数位（低精度，大范围）nums_b = generate_float_numbers(m_bits=3, e_bits=4)pos_b = [n for n in nums_b if 0 < n < 100]axes[1].scatter(pos_b, [0]*len(pos_b), c='blue', s=40, alpha=0.7)axes[1].axhline(y=0, color='black', linewidth=1)axes[1].set_title(f'方案B: 3位尾数+4位指数\n{len(pos_b)}个正数, 数字稀疏但范围大',                  fontsize=12, fontweight='bold')axes[1].set_xlabel('数值')axes[1].set_ylim(-0.5, 0.5); axes[1].set_yticks([])axes[1].set_xlim(-0.5, 100)plt.suptitle('精度 vs 范围的权衡 Precision vs Range Tradeoff', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# 可视化：溢出和下溢# Visualization: Overflow and Underflowfig, ax = plt.subplots(figsize=(14, 4))# 数轴ax.axhline(y=0.5, color='black', linewidth=2, xmin=0.05, xmax=0.95)# 区域标注regions = [    (0.05, 0.15, '#FF9999', '负溢出\nNegative\nOverflow'),    (0.15, 0.35, '#99FF99', '可表示的负数\nRepresentable\nNegative'),    (0.35, 0.45, '#FFFF99', '负下溢\nNegative\nUnderflow'),    (0.45, 0.55, '#FFFFFF', 'Zero'),    (0.55, 0.65, '#FFFF99', '正下溢\nPositive\nUnderflow'),    (0.65, 0.85, '#99FF99', '可表示的正数\nRepresentable\nPositive'),    (0.85, 0.95, '#FF9999', '正溢出\nPositive\nOverflow'),]for x1, x2, color, label in regions:    rect = mpatches.FancyBboxPatch((x1, 0.2), x2-x1, 0.6,        boxstyle="round,pad=0.02", facecolor=color, edgecolor='black', linewidth=1, alpha=0.5)    ax.add_patch(rect)    ax.text((x1+x2)/2, 0.5, label, ha='center', va='center', fontsize=9, fontweight='bold')# 标记关键点key_points = [    (0.15, '-Max', 'red'), (0.35, '-Min', 'orange'),    (0.5, '0', 'black'),    (0.65, '+Min', 'orange'), (0.85, '+Max', 'red'),]for x, label, color in key_points:    ax.plot(x, 0.5, 'v', color=color, markersize=12, zorder=5)    ax.text(x, 0.05, label, ha='center', va='center', fontsize=10, fontweight='bold', color=color)ax.set_xlim(0, 1); ax.set_ylim(-0.1, 1.0)ax.set_title('浮点数的可表示范围 - 溢出 (Overflow) 和下溢 (Underflow)', fontsize=14, fontweight='bold')ax.axis('off')plt.tight_layout()plt.show()

In [ ]:
# 交互式转换工具# Interactive Conversion Tooldef decimal_to_float_binary(value, m_bits=8, e_bits=4):    """Decimal to floating point binary (simplified)"""    if value == 0:        return [0]*m_bits, [0]*e_bits, "零"    is_negative = value < 0    abs_val = abs(value)    # 找到指数    exponent = 0    normalized = abs_val    if normalized >= 1.0:        while normalized >= 1.0:            normalized /= 2            exponent += 1    elif normalized < 0.5:        while normalized < 0.5:            normalized *= 2            exponent -= 1    # 现在 0.5 <= normalized < 1.0    # 转为二进制尾数    mantissa = [0]  # 符号位    temp = normalized    for i in range(1, m_bits):        threshold = 2 ** (-i)        if temp >= threshold:            mantissa.append(1)            temp -= threshold        else:            mantissa.append(0)    # 处理负数（取补码）    if is_negative:        # 取反+1        mantissa = [1 - b for b in mantissa]        carry = 1        for i in range(m_bits - 1, -1, -1):            total = mantissa[i] + carry            mantissa[i] = total % 2            carry = total // 2    # 指数转补码    exp_max = 2 ** (e_bits - 1) - 1    exp_min = -(2 ** (e_bits - 1))    overflow = ""    if exponent > exp_max:        overflow = "溢出！指数太大"        exponent = exp_max    elif exponent < exp_min:        overflow = "下溢！指数太小"        exponent = exp_min    if exponent >= 0:        exp_bits = []        temp_e = exponent        for i in range(e_bits):            exp_bits.insert(0, temp_e % 2)            temp_e //= 2    else:        # 负数补码        pos_e = abs(exponent)        exp_bits_pos = []        for i in range(e_bits):            exp_bits_pos.insert(0, pos_e % 2)            pos_e //= 2        exp_bits = [1 - b for b in exp_bits_pos]        carry = 1        for i in range(e_bits - 1, -1, -1):            total = exp_bits[i] + carry            exp_bits[i] = total % 2            carry = total // 2    return mantissa, exp_bits, overflow# 演示print("=" * 60)print("十进制 -> 浮点二进制 转换器")print("=" * 60)test_values = [13.0, 5.75, -6.5, 0.125, -0.375, 100.0, 0.001]fpc = FloatingPointConverter()for val in test_values:    mantissa, exponent, note = decimal_to_float_binary(val)    m_str = f"{mantissa[0]}.{''.join(map(str, mantissa[1:]))}"    e_str = ''.join(map(str, exponent))    # 验证    m_val, e_val, result = fpc.binary_float_to_decimal(mantissa, exponent)    print(f"\n  {val:>8} -> 尾数: {m_str}  指数: {e_str}")    print(f"           验证: {m_val} x 2^{e_val} = {result}")    if note:        print(f"           注意: {note}")    norm = "YES" if fpc.is_normalized(mantissa) else "NO"    print(f"           规格化: {norm}")

---## 八、总结 Summary### 核心概念回顾| 概念 | 要点 ||------|------|| **浮点数结构** | 数值 = 尾数 x 2^指数 || **尾数 Mantissa** | 二进制补码小数，决定精度 || **指数 Exponent** | 二进制补码整数，决定范围 || **规格化 (正数)** | 尾数形如 `0.1xxxxxx` || **规格化 (负数)** | 尾数形如 `1.0xxxxxx` || **浮点加法** | 对齐指数 -> 尾数相加 -> 规格化 || **浮点乘法** | 尾数相乘 -> 指数相加 -> 规格化 || **精度** | 由尾数位数决定 || **溢出** | 结果超出可表示的最大/最小值 || **下溢** | 结果太接近零，无法表示 || **舍入误差** | 有限位数导致精度损失（如 0.1） |### Cambridge 考试答题要点1. **转换题**：给定二进制浮点数，转换为十进制（或反过来）2. **规格化题**：判断是否规格化，若未规格化则进行规格化3. **运算题**：浮点数加法/减法的完整步骤4. **分析题**：解释增加尾数/指数位数对精度和范围的影响5. **误差题**：解释溢出、下溢、舍入误差的原因和影响

---## 九、练习 Exercises### 练习1：浮点数转换将以下二进制浮点数转换为十进制（8位尾数 + 4位指数）：| 尾数 | 指数 | 十进制 = ? ||------|------|-----------|| `0.1100000` | `0011` | ? || `0.1010000` | `0100` | ? || `1.0100000` | `0011` | ? || `0.1111000` | `1110` | ? |### 练习2：规格化判断以下尾数是否规格化。如果不是，请规格化并调整指数：| 尾数 | 指数 | 规格化？| 规格化后的尾数 | 规格化后的指数 ||------|------|--------|-------------|-------------|| `0.0110100` | `0101` | ? | ? | ? || `1.1010000` | `0011` | ? | ? | ? || `0.1001000` | `0010` | ? | ? | ? || `1.0011000` | `0100` | ? | ? | ? |

In [ ]:
# 练习1答案检查器 - 取消注释查看答案# Exercise 1 Answer Checkerdef check_exercise1():    fpc = FloatingPointConverter()    problems = [        ([0,1,1,0,0,0,0,0], [0,0,1,1], "0.1100000, 0011"),        ([0,1,0,1,0,0,0,0], [0,1,0,0], "0.1010000, 0100"),        ([1,0,1,0,0,0,0,0], [0,0,1,1], "1.0100000, 0011"),        ([0,1,1,1,1,0,0,0], [1,1,1,0], "0.1111000, 1110"),    ]    print("练习1 答案:")    print("-" * 50)    for mantissa, exponent, label in problems:        m_val, e_val, result = fpc.binary_float_to_decimal(mantissa, exponent)        print(f"  {label}")        print(f"  尾数={m_val}, 指数={e_val}, 数值={m_val} x 2^{e_val} = {result}")        print()# 取消注释查看答案:# check_exercise1()

### 练习3：浮点数加法使用 8 位尾数和 4 位指数，计算以下加法。写出完整步骤（对齐指数、尾数相加、规格化）：**(a)** 6.0 + 1.5- 6.0 = `0.1100000` x 2^`0011`- 1.5 = `0.1100000` x 2^`0001`**(b)** 10.0 + (-4.0)- 10.0 = `0.1010000` x 2^`0100`- -4.0 = `1.0000000` x 2^`0011`### 练习4：分析题**(a)** 一个浮点数系统有 8 位尾数和 4 位指数。如果将其改为 10 位尾数和 6 位指数（总位数从12变为16），对以下方面有什么影响？请分别解释：1. 精度 (Precision)2. 可表示的数值范围 (Range)3. 所需的存储空间 (Storage)**(b)** 解释为什么浮点数在接近零时分布更密集，而远离零时分布更稀疏。

### 练习5：Cambridge 风格考题**Question 1**: A computer uses a floating point representation with an 8-bit mantissa and a 4-bit exponent, both in two's complement.**(a)** Convert the floating point number with mantissa `01010000` and exponent `0101` to denary. Show your working. [3 marks]**(b)** Explain what is meant by normalisation of a floating point number. [2 marks]**(c)** The mantissa `00101000` with exponent `0110` represents an unnormalised number. Normalise this number, showing the new mantissa and exponent. [3 marks]**(d)** Explain one benefit of normalisation. [1 mark]**Question 2**: Two floating point numbers are to be added:- Number A: mantissa `01100000`, exponent `0011`- Number B: mantissa `01010000`, exponent `0001`**(a)** Show the steps required to add these two numbers. [4 marks]**(b)** Give the result as a normalised floating point number. [2 marks]**(c)** Explain what would happen if the result of an addition was too large to be stored. [2 marks]

In [ ]:
# 额外练习：浮点数表示的交互式可视化# Bonus: Interactive Floating Point Visualizationdef visualize_float(mantissa_str, exponent_str):    """可视化一个浮点数的各个部分"""    m_bits = [int(b) for b in mantissa_str.replace('.', '')]    e_bits = [int(b) for b in exponent_str]    fpc = FloatingPointConverter(len(m_bits), len(e_bits))    m_val, e_val, result = fpc.binary_float_to_decimal(m_bits, e_bits)    is_norm = fpc.is_normalized(m_bits)    fig, ax = plt.subplots(figsize=(12, 4))    # 画尾数    for i, bit in enumerate(mantissa_str):        if bit == '.':            ax.text(0.8 + i * 0.6, 0.5, '.', ha='center', va='center',                    fontsize=24, fontweight='bold', color='red')            continue        x = 0.5 + i * 0.6        color = '#FFD700' if i == 0 else ('#90EE90' if is_norm else '#FFCCCC')        rect = mpatches.FancyBboxPatch((x, 0.2), 0.5, 0.6,            boxstyle="round,pad=0.05", facecolor=color, edgecolor='black', linewidth=2)        ax.add_patch(rect)        ax.text(x + 0.25, 0.5, bit, ha='center', va='center', fontsize=16, fontweight='bold')    # 画指数    offset = len(mantissa_str) * 0.6 + 1    for i, bit in enumerate(exponent_str):        x = offset + i * 0.6        color = '#ADD8E6'        rect = mpatches.FancyBboxPatch((x, 0.2), 0.5, 0.6,            boxstyle="round,pad=0.05", facecolor=color, edgecolor='black', linewidth=2)        ax.add_patch(rect)        ax.text(x + 0.25, 0.5, bit, ha='center', va='center', fontsize=16, fontweight='bold')    # 结果    status = "规格化" if is_norm else "未规格化"    status_color = 'green' if is_norm else 'red'    info = f"尾数={m_val:.6f}  指数={e_val}  数值={result}  [{status}]"    ax.text(0.5, -0.2, info, fontsize=12, fontweight='bold', color=status_color)    ax.set_xlim(0, offset + len(exponent_str) * 0.6 + 1)    ax.set_ylim(-0.5, 1.2)    ax.set_title(f'浮点数可视化: {mantissa_str} | {exponent_str}', fontsize=14, fontweight='bold')    ax.axis('off')    plt.tight_layout()    plt.show()# 试试不同的浮点数print("可视化几个浮点数:\n")visualize_float("0.1101000", "0100")visualize_float("1.0110000", "0011")visualize_float("0.0101000", "0101")  # 未规格化

---**恭喜你完成了浮点数表示与运算的学习！**这是 A-Level Computer Science 中最具挑战性的话题之一。关键是要多练习转换和运算步骤，确保在考试中能准确、快速地完成计算。**复习建议**：1. 反复练习二进制小数和十进制之间的转换2. 牢记规格化条件：正数 `0.1xxx`，负数 `1.0xxx`3. 浮点加法的三步骤：对齐 -> 相加 -> 规格化4. 理解精度/范围的权衡关系5. 能够解释溢出、下溢和舍入误差的原因